# PubMed Pull
Viewing the pull from PubMed

## Setup

In [2]:
# Import all required packages
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from pymed import PubMed
import json
import os
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import nltk
from nltk.tokenize import sent_tokenize
import plotly.express as px

# Load environment variables
load_dotenv()

True

Import search_strategy

In [8]:
from search_strategy import (
    INCLUSION_CRITERIA,
    EXCLUSION_TERMS,
    DATE_FILTER,
    DATABASE_CONFIGS,
    CLEANING_RULES,
    QUERY_RESULTS,
)

In [12]:
# Access inclusion/exclusion terms
print(INCLUSION_CRITERIA["disease"])
print(EXCLUSION_TERMS)

# Get date range
start_date = DATE_FILTER["start_date"]
end_date = DATE_FILTER["end_date"]
print(start_date)
print(end_date)

['parkinson* disease', 'neurodegenerative disease']
['pathology', 'treatment', 'therapy', 'intervention', 'physiology', 'monitoring', 'biosensor', 'animal', 'plant', 'in vitro', 'molecular', 'protein', 'mice', 'parkinson* disease model', 'respiratory', 'resistance training', 'aggression']
2020-01-01
2025-12-31


## Functions

In [3]:
# Utility: safe emptiness check for arrays/iterables
def is_empty(obj):
    """Return True if obj is None or has zero length/size; safe for lists, pandas Series, numpy arrays, sparse matrices."""
    if obj is None:
        return True
    # Try length
    try:
        return len(obj) == 0
    except Exception:
        pass
    # Try numpy-like size attribute
    try:
        return getattr(obj, 'size', 0) == 0
    except Exception:
        return False
    
# Preview `results` as a pandas DataFrame
# This cell converts the `results` list (PubMed article objects) into a DataFrame
# and displays a concise preview (first N rows).

if 'results' not in globals() or is_empty(results):
    print("`results` is not defined or empty. Run the query cell above to populate `results`.")
else:
    def _article_to_flat_dict(article):
        # Try to leverage article.toJSON() when available
        try:
            raw = article.toJSON()
            if isinstance(raw, str):
                return json.loads(raw)
            if isinstance(raw, dict):
                return raw
        except Exception:
            pass

        # Fallback: extract common attributes safely
        return {
            "pubmed_id": getattr(article, "pubmed_id", None),
            "title": getattr(article, "title", None),
            "publication_date": str(getattr(article, "publication_date", "") or ""),
            "keywords": getattr(article, "keywords", None),
            "abstract": getattr(article, "abstract", None),
        }

    df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])


`results` is not defined or empty. Run the query cell above to populate `results`.


## Query

In [4]:
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

# This is the expanded query using the NLP-driven term rankings. n = 6,335
# Adding exclusion criteria based on scope of research question. 
query = '("parkinson* disease" OR "neurodegenerative disease") AND ' \
'("geospatial" OR "spatial dependence" OR "spatiotemporal" OR "geographic" OR "environment*" OR "atmospheric") ' \
'AND ("pollution" OR "chemical" OR "pesticide")' \
'NOT ("pathology" OR "treatment" OR "therapy" OR "intervention" OR "physiology" OR "monitoring" OR "biosensor") ' \
'NOT ("animal") ' \
'NOT ("parkinson* disease model" OR "respiratory" OR "resistance training" OR "aggression")' # n = 326, 138 in last 5 years

'''
Meeting with Margaret 
Include air and water pollution
Look at EMBASE

# include spatial analysis"
# eliminate editorials
# need a good reason for a recency cutoff
# air pollution/ or microplastic pollution/ or traffic pollution/ or water pollution/ or pollution/
#(pollution or pollutant or trichloroethylene or pesticide*).ti,ab.
# Dont exclude plant, molecular, protein, and in vitro at the screening stage


^^Get Malcolm and Lukas to review the pollution and geo terms to expand terms


'''

# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=500))
df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])

# Limit to last 5 years
df_results['publication_date'] = pd.to_datetime(df_results['publication_date'], format='ISO8601')
start_date = '2020-01-01'
df_filtered = df_results[df_results['publication_date'] >= start_date]

# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
n = 5
print('Total number of articles:', df_results.shape[0]) # get number of rows
print('Total articles in last 5 years:', df_filtered.shape[0]) # get number of rows after date filter
display(df_filtered[preview_cols].head(n))

NameError: name '_article_to_flat_dict' is not defined

## Cleaning

In [6]:
# Removing duplicates in title
df_filtered = df_filtered.drop_duplicates(subset=['title'])

# Removing if pubmed_id is empty
df_filtered = df_filtered.dropna(subset=['pubmed_id'])

# Remove if doi is missing
df_filtered = df_filtered.dropna(subset=['doi'])

# Preview cleaned df
n = 5
print('Total articles after cleaning:', df_filtered.shape[0]) # get number of rows after cleaning
display(df_filtered[preview_cols].head(n))

Total articles after cleaning: 153


,title,abstract,publication_date,keywords
0,Air pollution and disease progression in a Uni...,"Amyotrophic lateral sclerosis (ALS) is a rare,...",2025-11-25,"[ALSFRS-R, Air pollution, Amyotrophic lateral ..."
1,Inhibition of Mitochondrial Complex III Causes...,Environmental factors including chemical expos...,2025-11-24,[]
2,Air pollution and Parkinson's disease: A prosp...,Air pollution has been suggested as a potentia...,2025-11-21,"[Air Pollution, Nitrogen Dioxide, Parkinson’s ..."
3,Do microplastics play a role in the pathogenes...,The widespread presence of microplastics (MPs)...,2025-11-18,"[Alzheimer’s disease, Microplastics, Neuro-pat..."
4,Environmental toxins in neurodegeneration - a ...,As the global incidence of neurodegenerative d...,2025-11-18,"[Air pollution, Alzheimer’s disease, Amyotroph..."


## Export

In [ ]:
# Export cleaned results to CSV
output_csv_path = Path("pubmed_results_cleaned_2025.csv")
df_filtered.to_csv(output_csv_path, index=False)
print(f"Exported cleaned results to {output_csv_path.resolve()}")